# NLP semántico en español: embeddings, cosine similarity y BERTopic

Objetivo: usar **embeddings semánticos** para representar textos en español, medir **similitud coseno**, detectar temas con **BERTopic** y calcular un **coherence score**.

La idea es:
1. Ejecutar primero este ejemplo.
2. Entender qué hace cada bloque.
3. Sustituir después el dataframe por los datos de tu trabajo.

## 0. Instalación de librerías

In [ ]:
!pip install -q sentence-transformers bertopic gensim pandas scikit-learn umap hdbscan

## 1. Imports

In [6]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

from bertopic import BERTopic

from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

pd.set_option("display.max_colwidth", 200)

## 2. Dataset de ejemplo en español

El dataset mezcla textos sobre universidad, tecnología, salud y vivienda. Hay temas bastante claros, pero también documentos con zonas de solapamiento: tecnología en universidades, salud mental en estudiantes, vivienda para jóvenes, etc.

In [7]:
documents = [
    # Universidad / educación
    "Los estudiantes protestaron por el aumento de las tasas universitarias y reclamaron más ayudas para material y transporte.",
    "La universidad anunció un nuevo programa de becas destinado a alumnos con dificultades económicas y buen expediente académico.",
    "Profesores y estudiantes criticaron la falta de espacios de estudio durante el periodo de exámenes finales.",
    "El rector presentó un plan para modernizar las aulas mediante herramientas digitales e inteligencia artificial.",
    "Muchos alumnos compaginan trabajo y estudios debido al coste creciente de la matrícula y del alojamiento.",

    # Tecnología / inteligencia artificial
    "Varias empresas tecnológicas están incorporando modelos de inteligencia artificial para automatizar tareas administrativas.",
    "Expertos en ciberseguridad alertan sobre el aumento de ataques informáticos dirigidos a hospitales y universidades.",
    "Los avances recientes en procesamiento del lenguaje natural permiten desarrollar asistentes virtuales más precisos.",
    "Investigadores europeos trabajan en sistemas de inteligencia artificial capaces de detectar noticias falsas en redes sociales.",
    "Las nuevas herramientas de análisis de datos ayudan a detectar patrones ocultos en grandes volúmenes de información.",

    # Salud
    "Los médicos advirtieron sobre el incremento de problemas de ansiedad y estrés entre adolescentes y universitarios.",
    "El hospital central redujo las listas de espera gracias a un nuevo sistema digital de gestión de pacientes.",
    "Especialistas recomiendan mejorar los hábitos de sueño y reducir el uso nocturno del teléfono móvil.",
    "La campaña de vacunación logró aumentar la cobertura sanitaria en varias zonas rurales del país.",
    "La falta de descanso y la presión académica están afectando al bienestar emocional de muchos jóvenes.",

    # Vivienda / economía
    "El precio del alquiler continúa aumentando en las grandes ciudades, dificultando el acceso a la vivienda para los jóvenes.",
    "El gobierno aprobó nuevas ayudas destinadas a familias afectadas por la subida de las hipotecas.",
    "Muchos estudiantes comparten piso debido al elevado coste de la vivienda cerca de las universidades.",
    "Expertos inmobiliarios advierten de una desaceleración del mercado residencial durante el próximo año.",
    "La inflación y el encarecimiento de los suministros han reducido la capacidad de ahorro de muchas familias."
]

category = (
    ["universidad"] * 5 +
    ["tecnologia"] * 5 +
    ["salud"] * 5 +
    ["vivienda"] * 5
)

df = pd.DataFrame({
    "text": documents,
    "category": category
})

display(df)

,text,category
0,Los estudiantes protestaron por el aumento de las tasas universitarias y reclamaron más ayudas para material y transporte.,universidad
1,La universidad anunció un nuevo programa de becas destinado a alumnos con dificultades económicas y buen expediente académico.,universidad
2,Profesores y estudiantes criticaron la falta de espacios de estudio durante el periodo de exámenes finales.,universidad
3,El rector presentó un plan para modernizar las aulas mediante herramientas digitales e inteligencia artificial.,universidad
4,Muchos alumnos compaginan trabajo y estudios debido al coste creciente de la matrícula y del alojamiento.,universidad
5,Varias empresas tecnológicas están incorporando modelos de inteligencia artificial para automatizar tareas administrativas.,tecnologia
6,Expertos en ciberseguridad alertan sobre el aumento de ataques informáticos dirigidos a hospitales y universidades.,tecnologia
7,Los avances recientes en procesamiento del lenguaje natural permiten desarrollar asistentes virtuales más precisos.,tecnologia
8,Investigadores europeos trabajan en sistemas de inteligencia artificial capaces de detectar noticias falsas en redes sociales.,tecnologia
9,Las nuevas herramientas de análisis de datos ayudan a detectar patrones ocultos en grandes volúmenes de información.,tecnologia


## 3. Configuración

In [51]:
TEXT_COLUMN = "text"
GROUP_COLUMN = "category"

documents = df[TEXT_COLUMN].dropna().astype(str).tolist()

print(f"Número de documentos: {len(documents)}")

Número de documentos: 20


## 4. Embeddings semánticos

Usamos un modelo multilingüe de `sentence-transformers`, adecuado para textos en español.

Cada texto se transforma en un vector. La idea es que textos con significado parecido tengan vectores cercanos.

In [15]:
# Ten en cuenta que los modelos Transformer no requieren de lematización, quitar stopwords, ni tokenización previa. De hecho, estos procesos pueden eliminar información relevante para el modelo.
embedding_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

embeddings = embedding_model.encode(
    documents,
    show_progress_bar=True
)

print("Shape de embeddings:", embeddings.shape)

Batches: 100%|██████████| 1/1 [00:00<00:00,  5.70it/s]

Shape de embeddings: (20, 384)


## Cambiar el modelo de embeddings

`SentenceTransformers` permite utilizar modelos preentrenados descargados desde Hugging Face:

https://huggingface.co/models

Para usar otro modelo, consulta la documentación del mismo.
```

### Importante

Distintos modelos pueden producir resultados muy diferentes.

Algunas diferencias habituales:

- algunos modelos son más rápidos que otros,
- algunos funcionan mejor en español,
- algunos generan embeddings más precisos pero consumen más memoria.

Cambiar el modelo puede modificar completamente:

- la `cosine similarity`,
- el clustering,
- los topics detectados por `BERTopic`,
- el `coherence score`.

Por tanto, la elección del modelo de embeddings influye directamente en la representación semántica del corpus y en la calidad final del análisis.

## 5. Cosine similarity

La similitud coseno mide si dos vectores apuntan en una dirección parecida.

Aquí elegimos un documento y buscamos los documentos semánticamente más similares.

In [12]:
query_idx = 0

similarities = cosine_similarity(
    [embeddings[query_idx]],
    embeddings
)[0]

# Excluimos el propio documento con [1:6]
top_similar = similarities.argsort()[::-1][1:6]

print("DOCUMENTO ORIGINAL")
print("=" * 80)
print(f"Índice: {query_idx}")
print(f"Categoría: {df.iloc[query_idx][GROUP_COLUMN]}")
print(documents[query_idx])

print("\nDOCUMENTOS MÁS SIMILARES")
for idx in top_similar:
    print("\n" + "=" * 80)
    print(f"Índice: {idx}")
    print(f"Similitud coseno: {similarities[idx]:.3f}")
    print(f"Categoría: {df.iloc[idx][GROUP_COLUMN]}")
    print(documents[idx])

DOCUMENTO ORIGINAL
Índice: 0
Categoría: universidad
Los estudiantes protestaron por el aumento de las tasas universitarias y reclamaron más ayudas para material y transporte.

DOCUMENTOS MÁS SIMILARES

Índice: 4
Similitud coseno: 0.644
Categoría: universidad
Muchos alumnos compaginan trabajo y estudios debido al coste creciente de la matrícula y del alojamiento.

Índice: 17
Similitud coseno: 0.596
Categoría: vivienda
Muchos estudiantes comparten piso debido al elevado coste de la vivienda cerca de las universidades.

Índice: 10
Similitud coseno: 0.494
Categoría: salud
Los médicos advirtieron sobre el incremento de problemas de ansiedad y estrés entre adolescentes y universitarios.

Índice: 1
Similitud coseno: 0.490
Categoría: universidad
La universidad anunció un nuevo programa de becas destinado a alumnos con dificultades económicas y buen expediente académico.

Índice: 14
Similitud coseno: 0.487
Categoría: salud
La falta de descanso y la presión académica están afectando al bienestar

## 6. Matriz de similitud coseno

También podemos calcular la similitud entre todos los documentos y visualizarla como una matriz.

In [16]:
similarity_matrix = cosine_similarity(embeddings)

sim_df = pd.DataFrame(
    similarity_matrix,
    index=[f"doc_{i}" for i in range(len(documents))],
    columns=[f"doc_{i}" for i in range(len(documents))]
)

display(sim_df.round(2))

,doc_0,doc_1,doc_2,doc_3,doc_4,doc_5,doc_6,doc_7,doc_8,doc_9,doc_10,doc_11,doc_12,doc_13,doc_14,doc_15,doc_16,doc_17,doc_18,doc_19
doc_0,1.00,0.49,0.41,0.15,0.64,-0.10,0.22,0.03,-0.02,-0.04,0.49,-0.02,0.10,0.14,0.49,0.37,0.24,0.60,0.17,0.26
doc_1,0.49,1.00,0.33,0.42,0.52,0.13,0.19,0.16,0.16,0.16,0.38,0.15,0.10,0.06,0.38,0.11,0.29,0.46,0.08,0.16
doc_2,0.41,0.33,1.00,0.23,0.48,-0.01,0.17,0.12,0.03,0.01,0.43,0.16,0.17,-0.07,0.50,0.16,-0.07,0.42,0.23,0.23
doc_3,0.15,0.42,0.23,1.00,0.25,0.51,0.27,0.46,0.28,0.18,0.11,0.33,0.24,0.13,0.11,0.05,0.08,0.21,0.12,0.07
doc_4,0.64,0.52,0.48,0.25,1.00,0.01,0.23,0.17,0.06,0.06,0.43,0.10,0.20,0.19,0.50,0.55,0.18,0.78,0.18,0.30
doc_5,-0.10,0.13,-0.01,0.51,0.01,1.00,0.30,0.42,0.45,0.27,0.01,0.30,0.19,0.01,-0.00,-0.08,0.03,-0.01,0.07,-0.00
doc_6,0.22,0.19,0.17,0.27,0.23,0.30,1.00,0.23,0.42,0.29,0.42,0.21,0.25,0.18,0.17,0.17,0.04,0.22,0.14,-0.01
doc_7,0.03,0.16,0.12,0.46,0.17,0.42,0.23,1.00,0.32,0.27,-0.03,0.18,0.18,0.21,0.02,-0.03,0.02,0.01,-0.01,-0.08
doc_8,-0.02,0.16,0.03,0.28,0.06,0.45,0.42,0.32,1.00,0.45,0.07,0.22,0.08,0.05,0.06,0.01,-0.01,0.10,0.10,-0.08
doc_9,-0.04,0.16,0.01,0.18,0.06,0.27,0.29,0.27,0.45,1.00,0.02,0.25,0.08,0.16,-0.03,0.04,0.14,0.01,0.12,0.04


![BERTopic](BERTopic.png)

## 7. Topic modeling semántico con BERTopic

BERTopic no parte solo de frecuencias de palabras. Primero usa embeddings para representar los documentos semánticamente y después agrupa documentos similares.

In [50]:
topic_model = BERTopic(
    embedding_model=embedding_model,
    language="multilingual",  
    min_topic_size=2,  # número mínimo de documentos por tema
    verbose=True
)

topics, probs = topic_model.fit_transform(
    documents,
    embeddings
)

df["topic"] = topics

display(df[[TEXT_COLUMN, GROUP_COLUMN, "topic"]])

2026-05-07 00:21:37,600 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-07 00:21:37,915 - BERTopic - Dimensionality - Completed ✓
2026-05-07 00:21:37,932 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-07 00:21:38,097 - BERTopic - Cluster - Completed ✓
2026-05-07 00:21:38,147 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-07 00:21:38,189 - BERTopic - Representation - Completed ✓


,text,category,topic
0,Los estudiantes protestaron por el aumento de las tasas universitarias y reclamaron más ayudas para material y transporte.,universidad,0
1,La universidad anunció un nuevo programa de becas destinado a alumnos con dificultades económicas y buen expediente académico.,universidad,0
2,Profesores y estudiantes criticaron la falta de espacios de estudio durante el periodo de exámenes finales.,universidad,0
3,El rector presentó un plan para modernizar las aulas mediante herramientas digitales e inteligencia artificial.,universidad,1
4,Muchos alumnos compaginan trabajo y estudios debido al coste creciente de la matrícula y del alojamiento.,universidad,0
5,Varias empresas tecnológicas están incorporando modelos de inteligencia artificial para automatizar tareas administrativas.,tecnologia,1
6,Expertos en ciberseguridad alertan sobre el aumento de ataques informáticos dirigidos a hospitales y universidades.,tecnologia,1
7,Los avances recientes en procesamiento del lenguaje natural permiten desarrollar asistentes virtuales más precisos.,tecnologia,1
8,Investigadores europeos trabajan en sistemas de inteligencia artificial capaces de detectar noticias falsas en redes sociales.,tecnologia,1
9,Las nuevas herramientas de análisis de datos ayudan a detectar patrones ocultos en grandes volúmenes de información.,tecnologia,1


![KeyBERTInspired](KeyBertInspired.png)

[BERTopic](https://maartengr.github.io/BERTopic/algorithm/algorithm.html) permite realizar una implementación más detallada. En el siguiente código se muestra una implementación más avanzada.


In [73]:
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer

#  Reduce dimensionality
umap_model = UMAP(
    n_neighbors=3,   # Número de vecinos para construir el grafo de proximidad en el espacio de alta dimensión. Un valor más alto puede capturar mejor la estructura global, mientras que un valor más bajo se enfoca en la estructura local.
    n_components=5,   # Número de dimensiones a las que se reducirá el espacio de embeddings. Un valor más bajo puede facilitar la visualización y el clustering, pero también puede perder información relevante.
    min_dist=0.0,      
    metric="cosine",  
    random_state=42
)

# Cluster reduced embeddings
hdbscan_model = HDBSCAN(
    min_cluster_size=2,   # Número mínimo de documentos por tema. Un valor más bajo puede generar más temas, pero también puede incluir temas menos coherentes.
    min_samples=2,        # Controla lo estricto que es HDBSCAN para considerar que una zona es realmente un cluster (en función de la densidad de muestras). Un valor más bajo puede generar más clusters, pero también puede incluir clusters menos coherentes.
    metric="euclidean",   # Métrica de distancia para calcular la similitud entre documentos
    cluster_selection_method="eom", # Método de selección de clusters "excess of mass" que prioriza clusters densos y bien definidos.
    prediction_data=True
)

# Tokenize topics
vectorizer_model = CountVectorizer(
    stop_words=None
)

# Create topic representation
ctfidf_model = ClassTfidfTransformer()

# All steps together
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    language="multilingual",
    verbose=True
)

topics, probs = topic_model.fit_transform(
    documents,
    embeddings
)

df["topic"] = topics

display(df[[TEXT_COLUMN, GROUP_COLUMN, "topic"]])

2026-05-07 01:00:28,659 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-07 01:00:28,699 - BERTopic - Dimensionality - Completed ✓
2026-05-07 01:00:28,700 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-07 01:00:28,706 - BERTopic - Cluster - Completed ✓
2026-05-07 01:00:28,708 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-07 01:00:28,716 - BERTopic - Representation - Completed ✓


,text,category,topic
0,Los estudiantes protestaron por el aumento de las tasas universitarias y reclamaron más ayudas para material y transporte.,universidad,0
1,La universidad anunció un nuevo programa de becas destinado a alumnos con dificultades económicas y buen expediente académico.,universidad,0
2,Profesores y estudiantes criticaron la falta de espacios de estudio durante el periodo de exámenes finales.,universidad,0
3,El rector presentó un plan para modernizar las aulas mediante herramientas digitales e inteligencia artificial.,universidad,1
4,Muchos alumnos compaginan trabajo y estudios debido al coste creciente de la matrícula y del alojamiento.,universidad,0
5,Varias empresas tecnológicas están incorporando modelos de inteligencia artificial para automatizar tareas administrativas.,tecnologia,1
6,Expertos en ciberseguridad alertan sobre el aumento de ataques informáticos dirigidos a hospitales y universidades.,tecnologia,1
7,Los avances recientes en procesamiento del lenguaje natural permiten desarrollar asistentes virtuales más precisos.,tecnologia,1
8,Investigadores europeos trabajan en sistemas de inteligencia artificial capaces de detectar noticias falsas en redes sociales.,tecnologia,1
9,Las nuevas herramientas de análisis de datos ayudan a detectar patrones ocultos en grandes volúmenes de información.,tecnologia,1


## 8. Información general de topics

In [74]:

topic_info = topic_model.get_topic_info()
display(topic_info)

,Topic,Count,Name,Representation,Representative_Docs
0,0,12,0_de_la_el_los,"[de, la, el, los, del, las, al, estudiantes, muchos, falta]","[La falta de descanso y la presión académica están afectando al bienestar emocional de muchos jóvenes., Muchos estudiantes comparten piso debido al elevado coste de la vivienda cerca de las univer..."
1,1,8,1_de_en_inteligencia_artificial,"[de, en, inteligencia, artificial, el, las, herramientas, detectar, un, para]","[El hospital central redujo las listas de espera gracias a un nuevo sistema digital de gestión de pacientes., Las nuevas herramientas de análisis de datos ayudan a detectar patrones ocultos en gra..."


## 9. Palabras principales por topic

In [71]:
unique_topics = sorted(set(topics))

for topic_id in unique_topics:
    if topic_id == -1:
        continue

    print("\n" + "=" * 80)
    print(f"TOPIC {topic_id}")

    words = topic_model.get_topic(topic_id)
    topic_words = [word for word, score in words[:10]]

    print(topic_words)


TOPIC 0
['de', 'la', 'el', 'los', 'del', 'las', 'al', 'estudiantes', 'muchos', 'falta']

TOPIC 1
['de', 'en', 'inteligencia', 'artificial', 'el', 'las', 'herramientas', 'detectar', 'un', 'para']


In [34]:
words = topic_model.get_topic(1)
print(words)

[('de', np.float64(0.17059286259283524)), ('en', np.float64(0.13264725889718904)), ('inteligencia', np.float64(0.10064790338918375)), ('artificial', np.float64(0.10064790338918375)), ('el', np.float64(0.08876915265628933)), ('las', np.float64(0.07958835533831342)), ('herramientas', np.float64(0.07391679754282016)), ('detectar', np.float64(0.07391679754282016)), ('un', np.float64(0.06709860225945584)), ('para', np.float64(0.0622930523123512))]


## 10. Coherence score

El coherence score intenta medir si las palabras principales de cada topic son coherentes entre sí.

La métrica más usada para ello es la $C_v$ ([Exploring the Space of Topic Coherence Measures](https://dl.acm.org/doi/10.1145/2684822.2685324))

#### Cálculo del score coherence global

In [36]:
# Tokenización sencilla para clase.
# Si el proyecto lo requiere, se puede mejorar con limpieza, stopwords o lematización.
tokenized_docs = [doc.lower().split() for doc in documents]
dictionary = Dictionary(tokenized_docs) # Se crea un diccionario de Gensim a partir de los documentos tokenizados, lo que permite mapear cada palabra a un ID único.

topics_for_coherence = []

for topic_id in unique_topics:
    if topic_id == -1:
        continue

    words = topic_model.get_topic(topic_id)
    topic_words = [word for word, score in words[:10]] # Solo las 10 palabras más representativas del topic
    topics_for_coherence.append(topic_words)

if len(topics_for_coherence) > 0:
    coherence_model = CoherenceModel(
        topics=topics_for_coherence, # Las listas de palabras representativas de cada topic se pasan como input para calcular la coherencia.
        texts=tokenized_docs, # Los textos tokenizados se pasan para calcular la coherencia basada en la co-ocurrencia de palabras en los documentos.
        dictionary=dictionary, # El diccionario de Gensim se pasa para mapear las palabras a sus IDs y calcular la coherencia correctamente.
        coherence="c_v" # La medida de coherencia "c_v" es una de las más utilizadas y combina la co-ocurrencia de palabras con la distancia semántica entre ellas.
    )

    coherence = coherence_model.get_coherence()
    # Cálculo de la coherencia global del modelo de topics utilizando la medida "c_v"
    print(f"Coherence Score: {coherence:.4f}")  # Un valor más alto indica que las palabras del topic tienden a aparecer juntas en los documentos, lo que sugiere un tema más coherente.
else:
    print("No hay topics suficientes para calcular coherence score.")

Coherence Score: 0.2599


#### Cálculo del score coherence por tema

In [37]:
tokenized_docs = [doc.lower().split() for doc in documents]

# Diccionario de Gensim:
# palabra -> ID único
dictionary = Dictionary(tokenized_docs)

print("=" * 80)
print("COHERENCE SCORE POR TOPIC")
print("=" * 80)

for topic_id in unique_topics:

    # Ignorar outliers de BERTopic
    if topic_id == -1:
        continue

    # Obtener palabras representativas del topic
    words = topic_model.get_topic(topic_id)

    # Seleccionar las 10 palabras más importantes
    topic_words = [word for word, score in words[:10]]

    # Mostrar palabras del topic
    print(f"\nTopic {topic_id}")
    print("-" * 40)
    print("Palabras representativas:")
    print(topic_words)

    # El coherence model espera una lista de topics,
    # aunque aquí solo evaluemos uno
    coherence_model = CoherenceModel(
        topics=[topic_words],
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence="c_v"
    )

    # Calcular coherence del topic individual
    coherence = coherence_model.get_coherence()

    print(f"Coherence Score: {coherence:.4f}")

COHERENCE SCORE POR TOPIC

Topic 0
----------------------------------------
Palabras representativas:
['de', 'la', 'el', 'los', 'del', 'las', 'al', 'estudiantes', 'muchos', 'falta']
Coherence Score: 0.2441

Topic 1
----------------------------------------
Palabras representativas:
['de', 'en', 'inteligencia', 'artificial', 'el', 'las', 'herramientas', 'detectar', 'un', 'para']
Coherence Score: 0.2757


In [31]:
dictionary

## 11. Cruce topic × variable de interés

Este bloque permite responder preguntas como:

- ¿Qué topics aparecen más en cada periódico?
- ¿Qué topics se asocian a mejores o peores valoraciones?
- ¿Qué topics aparecen más en cada género musical?
- ¿Qué topics están sobrerrepresentados en noticias clickbait?

In [38]:
if GROUP_COLUMN in df.columns:
    topic_by_group = (
        df.groupby(GROUP_COLUMN)["topic"]
        .value_counts(normalize=True)
        .rename("proportion")
        .reset_index()
        .sort_values([GROUP_COLUMN, "proportion"], ascending=[True, False])
    )

    display(topic_by_group)
else:
    print(f"La columna {GROUP_COLUMN} no existe en el dataframe.")

,category,topic,proportion
0,salud,0,0.6
1,salud,1,0.4
2,tecnologia,1,1.0
3,universidad,0,0.8
4,universidad,1,0.2
5,vivienda,0,1.0


## 12. Documentos representativos por topic

Para interpretar un topic no basta con mirar las palabras principales. Conviene leer varios documentos reales asignados a ese topic.

In [75]:
for topic_id in unique_topics:
    if topic_id == -1:
        continue

    print("\n" + "=" * 80)
    print(f"TOPIC {topic_id}")

    topic_docs = df[df["topic"] == topic_id]

    for i, (_, row) in enumerate(topic_docs.head(3).iterrows(), start=1):
        print(f"\nDocumento {i} | {GROUP_COLUMN}: {row.get(GROUP_COLUMN, 'NA')}")
        print(row[TEXT_COLUMN][:500])


TOPIC 0

Documento 1 | category: universidad
Los estudiantes protestaron por el aumento de las tasas universitarias y reclamaron más ayudas para material y transporte.

Documento 2 | category: universidad
La universidad anunció un nuevo programa de becas destinado a alumnos con dificultades económicas y buen expediente académico.

Documento 3 | category: universidad
Profesores y estudiantes criticaron la falta de espacios de estudio durante el periodo de exámenes finales.

TOPIC 1

Documento 1 | category: universidad
El rector presentó un plan para modernizar las aulas mediante herramientas digitales e inteligencia artificial.

Documento 2 | category: tecnologia
Varias empresas tecnológicas están incorporando modelos de inteligencia artificial para automatizar tareas administrativas.

Documento 3 | category: tecnologia
Expertos en ciberseguridad alertan sobre el aumento de ataques informáticos dirigidos a hospitales y universidades.


## 13. Visualizaciones de BERTopic

Estas visualizaciones suelen funcionar mejor con datasets más grandes (con dos cluster no podrás visualizar)

In [62]:
!pip install nbformat
!pip install notebook ipykernel jupyter

In [76]:

topic_model.visualize_topics()
topic_model.visualize_barchart()
topic_model.visualize_heatmap()


ValueError: zero-size array to reduction operation maximum which has no identity